In [ ]:
%config InlineBackend.figure_format = 'retina'

import ast
import json
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import seaborn as sns
import zstandard as zstd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.manifold import TSNE
from tqdm.notebook import tqdm

import difflib
import numpy as np
import scipy.stats as stats

from sklearn.base import clone
from sklearn.experimental import enable_halving_search_cv
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, HalvingRandomSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from xgboost import XGBClassifier

COLOR_SUCCESS = "#009E73"
COLOR_FAIL = "#D55E00"

# Load data for QA task
def load_qa_data(dataset, model_size):
    input_path = f"../data/{dataset}/processed_profile_results_llamacpp_qwen3_{model_size}b_judged.csv"
    input_df = pd.read_csv(input_path)
    is_correct = input_df["agent_output_eval"] == "CORRECT"
    input_df["agent_output_is_correct"] = is_correct
    if model_size == "30":
        llm_eval_file = f"../experiment_data/processed/{dataset}/llm_{dataset}_results_qwen3_30b_judged.csv"
        llm_eval_col = "qwen3:30b-a3b-instruct-2507-q4_K_M_eval"
    else:
        llm_eval_file = f"../experiment_data/processed/{dataset}/llm_{dataset}_results{'' if dataset == 'frames' else '_qwen3_' + model_size + 'b'}_judged.csv"
        llm_eval_col = f"qwen3:{model_size}b_eval"
    llm_eval = pd.read_csv(llm_eval_file)
    is_llm_correct = llm_eval[llm_eval_col] == "CORRECT"

    # Only consider traces where the LLM doesn't already know the answer
    all_logprobs = input_df["logprobs"].apply(eval).to_list()
    all_tokens = input_df["tokens"].apply(eval).to_list()
    all_logprobs = [all_logprobs[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
    all_tokens = [all_tokens[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
    is_correct = [is_correct[i] for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
    valid_idx = [i for i in range(len(is_llm_correct)) if not is_llm_correct[i]]
    all_data = input_df.iloc[valid_idx].copy()

    max_steps = 10
    exit_energy = all_data["step_0_energy_total_mWh"].copy()
    for step in range(max_steps):
        all_data[f"exit_energy_step_{step}"] = exit_energy
        try:
            step_energy = all_data[f"step_{step + 1}_energy_total_mWh"].fillna(0)
        except:
            print(f"Step {step + 1} is not found.")
            break
        exit_energy += step_energy

    all_data["total_energy"] = exit_energy

    return (all_logprobs, all_tokens, is_correct, valid_idx, all_data)

In [ ]:
# Make sure to unzip the data in experiment_data/

# Load Q&A data
qa_data = {}
for size in ["30", "1.7"]:
    for dataset in ["frames", "simpleqa"]:
        qa_data[(size, dataset)] = load_qa_data(dataset, size)

# Load SWE-Bench data
swebench_df = pd.read_csv(f"../experiment_data/processed/swebench/processed_profile_results_llamacpp_qwen3_coder_30b.csv")

swebench_is_correct = swebench_df["agent_output_eval"] == "CORRECT"
swebench_df["agent_output_is_correct"] = swebench_is_correct
swebench_num_tasks = len(swebench_df)
swebench_valid_idx = list(range(swebench_num_tasks))
swebench_all_logprobs = swebench_df["logprobs"].apply(eval).to_list()
swebench_all_tokens = swebench_df["tokens"].apply(eval).to_list()

In [ ]:
# Table 1
# Summary of Qwen3-30B-A3B and Qwen3-1.7B’s default agent performance on FRAMES and SimpleQA.

for size in ["30", "1.7"]:
    for dataset in ["frames", "simpleqa"]:
        print("="*10)
        print(f"Model: Qwen3-{size}b | Dataset: {dataset}")
        all_logprobs, all_tokens, is_correct, valid_idx, all_data = qa_data[(size, dataset)]

        # Analyze success rate and energy usage
        print(f"Task Accuracy: {round(np.mean(is_correct), 3)}")
        max_steps = 10

        exit_energy = all_data["step_0_energy_total_mWh"].copy()
        for step in range(max_steps):
            all_data[f"exit_energy_step_{step}"] = exit_energy
            try:
                step_energy = all_data[f"step_{step + 1}_energy_total_mWh"].fillna(0)
            except:
                print(f"Step {step + 1} is not found.")
                break
            exit_energy += step_energy

        all_data["total_energy"] = exit_energy

        fails = all_data[~all_data["agent_output_is_correct"]]
        successes = all_data[all_data["agent_output_is_correct"]]
        for label, data in [("All", all_data), ("Fails", fails), ("Successes", successes)]:
            total_energy = data["total_energy"]
            mean_energy = total_energy.mean()
            sem = stats.sem(total_energy)  # standard error of the mean
            confidence = 0.95
            ci = stats.t.interval(confidence, df=len(total_energy)-1, loc=mean_energy, scale=sem)
            print(f"Energy stats for {label}: n {len(total_energy)} | mean {mean_energy:.1f} | median {total_energy.median():.1f} | std {total_energy.std():.1f} | sem {sem:.1f} | 95% CI [{ci[0]:.1f}, {ci[1]:.1f}]")

        print(f"Average duration: {all_data['duration_sec'].mean()}")

        output_tokens = all_data["step_0_llm.token_count.completion"].copy()
        for step in range(1, max_steps):
            output_tokens += all_data[f"step_{step}_llm.token_count.completion"].fillna(0)
        print(f"Average output tokens: {output_tokens.mean()}")

        input_tokens = all_data[[f"step_{step}_llm.token_count.prompt" for step in range(max_steps)]].max(axis=1)
        print(f"Average input tokens: {input_tokens.mean()}")


In [ ]:
# Figure 2: Histogram of the number of agent steps

for size in ["30", "1.7"]:
    for dataset in ["frames", "simpleqa"]:
        print("="*10)
        print(f"Model: Qwen3-{size}b | Dataset: {dataset}")
        all_logprobs, all_tokens, is_correct, valid_idx, all_data = qa_data[(size, dataset)]

        # Display barplot of number of successes/failures by number of steps taken to finish the task
        incorrect_steps = [0] * 11
        correct_steps = [0] * 11
        for i, logprobs in enumerate(all_logprobs):
            n = len(logprobs)
            if is_correct[i]:
                correct_steps[n - 1] += 1
            else:
                incorrect_steps[n - 1] += 1

        plt.figure(figsize=(6, 4))
        ax = plt.gca()
        ax.set_axisbelow(True)
        ax.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.6)
        for i in range(1, 12):
            plt.bar(i - 0.2, correct_steps[i - 1], width=0.4, color=COLOR_SUCCESS, label="Success" if i == 1 else "", edgecolor="black", linewidth=0.5)
            plt.bar(i + 0.2, incorrect_steps[i - 1], width=0.4, color=COLOR_FAIL, label="Fail" if i == 1 else "", edgecolor="black", linewidth=0.5)
        plt.xticks(list(range(1, 12)))

        ax.tick_params(axis='both', labelsize=20)
        ax.legend(fontsize=20)
        plt.tight_layout()
        plt.show()

In [ ]:
# Figure 3: Cumulative energy contribution

for size in ["30", "1.7"]:
    for dataset in ["frames", "simpleqa"]:
        print("="*10)
        print(f"Model: Qwen3-{size}b | Dataset: {dataset}")
        all_logprobs, all_tokens, is_correct, valid_idx, all_data = qa_data[(size, dataset)]

        fails = all_data[~all_data["agent_output_is_correct"]]
        successes = all_data[all_data["agent_output_is_correct"]]

        plt.figure(figsize=(6, 4))
        ax = plt.gca()
        ax.set_axisbelow(True)
        ax.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.6)

        for data, label, color in [(successes, "Success", COLOR_SUCCESS), (fails, "Fail", COLOR_FAIL)]:
            total_used_energy = data["total_energy"].sum()
            step_energy_contrib = []
            for step in range(max_steps):
                step_energy_contrib.append(data[f"exit_energy_step_{step}"].fillna(0).sum())

            step_energy_contrib_pct = [100 * energy / total_used_energy for energy in step_energy_contrib]
            step_energy_contrib_pct.append(100)
            plt.plot(np.arange(1, 12), step_energy_contrib_pct, label=label, color=color, marker="o")

        plt.xticks(np.arange(1, 12))
        plt.ylim(0, 105)

        ax.tick_params(axis='both', labelsize=20)
        ax.legend(fontsize=20)

        plt.tight_layout()
        plt.show()

In [ ]:
# Table 2: Summary of Qwen3-Coder-30B-A3B’s performance

# Analyze success rate and energy usage
print(f"Task Accuracy: {round(np.mean(swebench_is_correct), 2)}")

max_steps = 60

exit_energy = swebench_df["step_0_energy_total_mWh"].copy()
for step in range(max_steps):
    swebench_df[f"exit_energy_step_{step}"] = exit_energy
    step_energy = swebench_df[f"step_{step + 1}_energy_total_mWh"].fillna(0)
    exit_energy += step_energy

swebench_df["total_energy"] = exit_energy

swebench_fails = swebench_df[~swebench_df["agent_output_is_correct"]]
swebench_successes = swebench_df[swebench_df["agent_output_is_correct"]]
for label, data in [("All", swebench_df), ("Fails", swebench_fails), ("Successes", swebench_successes)]:
    total_energy = data["total_energy"]
    mean_energy = total_energy.mean()
    sem = stats.sem(total_energy)  # standard error of the mean
    confidence = 0.95
    ci = stats.t.interval(confidence, df=len(total_energy)-1, loc=mean_energy, scale=sem)
    print(f"{label}: n {len(total_energy)} | mean {mean_energy:.1f} | median {total_energy.median():.1f} | std {total_energy.std():.1f} | sem {sem:.1f} | 95% CI [{ci[0]:.1f}, {ci[1]:.1f}]")


output_tokens = swebench_df["step_0_llm.token_count.completion"].copy()
for step in range(1, max_steps):
    output_tokens += swebench_df[f"step_{step}_llm.token_count.completion"].fillna(0)

print(f"Avg output token: {output_tokens.mean()}")
print("Avg output token for fail", output_tokens[~swebench_df["agent_output_is_correct"]].mean())
print("Avg output token for successes", output_tokens[swebench_df["agent_output_is_correct"]].mean())

input_tokens = swebench_df[[f"step_{step}_llm.token_count.prompt" for step in range(max_steps)]].max(axis=1)
print("Avg input token", input_tokens.mean())
print("Avg input token for fail", input_tokens[~swebench_df["agent_output_is_correct"]].mean())
print("Avg input token for successes", input_tokens[swebench_df["agent_output_is_correct"]].mean())

In [ ]:
# Figure 4: Top 10 smallest logprobs for QA

for size in ["30", "1.7"]:
    for dataset in ["frames", "simpleqa"]:
        print("="*10)
        print(f"Model: Qwen3-{size}b | Dataset: {dataset}")
        all_logprobs, all_tokens, is_correct, valid_idx, all_data = qa_data[(size, dataset)]
        def plot_min_logprobs(
            start_step,
            end_step,
            num_logprob=10,
            exp=False,
            fit=False,
            normalize=False,
            trace_type=None,
            save=None,
        ):
            assert start_step <= end_step, "Start step must not be greater than end step"
            plt.figure(figsize=(16, 4))
            cnt = 0

            step_count = {}
            step_success_count = {}
            for i, logprobs in enumerate(all_logprobs):
                n = len(logprobs)
                step_count[n] = step_count.get(n, 0) + 1
                if is_correct[i]:
                    step_success_count[n] = step_success_count.get(n, 0) + 1

            max_steps = max(step_count.keys())
            for num_step in range(max_steps, 0, -1):
                step_count[num_step] = step_count.get(num_step, 0) + step_count.get(num_step + 1, 0)
                step_success_count[num_step] = step_success_count.get(num_step, 0) + step_success_count.get(num_step + 1, 0)

            for i, logprobs in enumerate(all_logprobs):
                if trace_type is not None:
                    if is_correct[i] != trace_type:
                        continue
                cnt += 1
                color = COLOR_SUCCESS if is_correct[i] else COLOR_FAIL
                style = "-" if not fit else "dotted"

                for step, step_logprobs in enumerate(logprobs[start_step-1:end_step]):
                    if exp:
                        step_logprobs = np.exp(step_logprobs)

                    line = np.sort(step_logprobs)[:num_logprob]

                    alpha_correct = 1 - step_success_count[start_step + step] / step_count[start_step + step]
                    alpha = alpha_correct if is_correct[i] else 1 - alpha_correct
                    alpha = max(alpha, 0.3)
                    start = step * num_logprob
                    plt.plot(range(start, start + len(line)), line, color=color, alpha=alpha, linestyle=style)

            ax = plt.gca()
            num_steps = end_step - start_step + 1
            xticks = [(step * num_logprob + num_logprob // 2) for step in range(num_steps)]
            xlabels = [f"{step + 1}" for step in range(num_steps)]
            ax.set_xticks(xticks)
            ax.set_xticklabels(xlabels)
            ax.set_xlim((-1, num_steps * num_logprob))

            ax.tick_params(axis='both', labelsize=24)
            plt.grid(True, axis="y", linestyle="--", alpha=0.5)
            plt.tight_layout()

            custom_legend = [
                Line2D([0], [0], color=COLOR_SUCCESS, lw=2, label="Success"),
                Line2D([0], [0], color=COLOR_FAIL, lw=2, label="Fail"),
            ]
            plt.legend(handles=custom_legend, fontsize=24, labelspacing=0.1, frameon=False, bbox_to_anchor=(1.0, -0.05), loc="lower right")
            plt.show()
        
        plot_min_logprobs(1, 10, num_logprob=10, exp=False, fit=False, normalize=False, trace_type=None)

In [ ]:
# Figure 5: Histogram of number of agent steps for coding

plt.figure(figsize=(6, 3))
plt.hist([len(lp) for i, lp in enumerate(swebench_all_logprobs) if not swebench_is_correct[i]], bins=40, color=COLOR_FAIL, alpha=1.0, label="Fail", edgecolor="black", linewidth=0.5)
plt.hist([len(lp) for i, lp in enumerate(swebench_all_logprobs) if swebench_is_correct[i]], bins=20, color=COLOR_SUCCESS, alpha=1.0, label="Success",  edgecolor="black", linewidth=0.5)
ax = plt.gca()
ax.set_axisbelow(True)
ax.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.6)
# plt.xlabel("Number of steps")
# plt.ylabel("Count")
plt.xticks(np.arange(0, 101, 10))
plt.vlines(60, 0, 70, color="gray", linestyles="--")
plt.ylim(0, 61)
plt.yticks(np.arange(0, 61, step=10))

plt.annotate(
    "Cut-off",
    xy=(62, 34),
    xytext=(62, 34),
    # arrowprops=dict(arrowstyle="->"),
    bbox=dict(facecolor='none', edgecolor='none')
)

plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 6: Cumulative energy contributation for coding

plt.figure(figsize=(6, 3))
ax = plt.gca()
ax.set_axisbelow(True)
ax.grid(axis="both", linestyle="--", linewidth=0.8, alpha=0.6)

for data, label, color in [(swebench_successes, "Success", COLOR_SUCCESS), (swebench_fails, "Fail", COLOR_FAIL)]:
    total_used_energy = data["total_energy"].sum()
    step_energy_contrib = []
    for step in range(max_steps):
        step_energy_contrib.append(data[f"exit_energy_step_{step}"].fillna(0).sum())

    step_energy_contrib_pct = [100 * energy / total_used_energy for energy in step_energy_contrib]
    plt.plot(np.arange(1, max_steps + 1), step_energy_contrib_pct, label=label, color=color, marker=".")

plt.xticks([1] + list(np.arange(10, max_steps + 1, 10)))
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Figure 7: Top 10 smallest logprobs for coding

def plot_min_logprobs(
    start_step,
    end_step,
    num_logprob=10,
    exp=False,
    trace_type=None,
):
    assert start_step <= end_step, "Start step must not be greater than end step"
    plt.figure(figsize=(16, 4))
    cnt = 0

    for i, logprobs in enumerate(swebench_all_logprobs):
        if trace_type is not None:
            if swebench_is_correct[i] != trace_type:
                continue
        cnt += 1
        color = COLOR_SUCCESS if swebench_is_correct[i] else COLOR_FAIL
        style = "-"
        alpha = 0.8 if swebench_is_correct[i] else 0.3

        for step, step_logprobs in enumerate(logprobs[start_step-1:min(len(logprobs), end_step)]):
            if exp:
                step_logprobs = np.exp(step_logprobs)

            line = np.sort(step_logprobs)[:num_logprob]

            start = step * num_logprob + 1
            plt.plot(range(start, start + len(line)), line, color=color, alpha=alpha, linestyle=style)

    ax = plt.gca()
    num_steps = end_step - start_step + 1
    xticks = [(step * num_logprob + num_logprob // 2) for step in range(num_steps)]
    xlabels = [f"{start_step + step}" for step in range(num_steps)]
    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels)
    ax.set_xlim((-1, num_steps * num_logprob))
    ax.tick_params(axis='both', labelsize=24)

    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    
    custom_legend = [
        Line2D([0], [0], color=COLOR_SUCCESS, lw=4, label="Success"),
        Line2D([0], [0], color=COLOR_FAIL, lw=4, label="Fail"),
    ]
    plt.legend(handles=custom_legend, fontsize=28, ncol=2, labelspacing=0.1, columnspacing=0.5, handlelength=1.5, frameon=False, borderpad=0.0, bbox_to_anchor=(1.01, 1.18), loc="upper right")
    plt.show()

for i in range(4):
    plot_min_logprobs(i * 10 + 1, i * 10 + 10, num_logprob=10, exp=False)

In [ ]:
# Figure 8: AUC-ROC for QA

fpr_targets = [0.05 * i for i in range(21)]

def lcs(a: str, b: str) -> str:
    matcher = difflib.SequenceMatcher(None, a, b)
    match = matcher.find_longest_match(0, len(a), 0, len(b))
    if match.size == 0:
        return ""
    return a[match.a: match.a + match.size]

def normalize(arr):
    return (arr - np.mean(arr)) / np.std(arr)

# ------------------ 1) Prepare training data ------------------ #
def extract_features(logprobs_data, tokens_data, labels, clf_step, num_logprobs=10):
    X_features = []
    y = []
    valid = []

    for i, (logprobs, tokens) in enumerate(zip(logprobs_data, tokens_data)):
        if len(logprobs) <= clf_step:
            continue
        feature_vector = []

        probs = [np.exp(lp) for lp in logprobs[:clf_step]]
        min_logprobs = [v for p in probs for v in np.sort(p)[:num_logprobs]]
        feature_vector.extend(min_logprobs)

        # Token features
        num_tokens = [len(lp) for lp in logprobs[:clf_step]]
        feature_vector.extend(num_tokens)

        num_thought_tokens = [(step_tokens.index("<code") if "<code" in step_tokens else len(step_tokens)) for step_tokens in tokens[:clf_step]]
        feature_vector.extend(num_thought_tokens)

        # Length of longest common substring between current step and previous step
        for j in range(max(clf_step - 1, 1), clf_step):
            cur_gen = "".join(tokens[j])
            prev_gen = "".join(tokens[j - 1])
            feature_vector.append(len(lcs(cur_gen, prev_gen)) / len(cur_gen))

        X_features.append(feature_vector)
        y.append(int(labels[i]))
        valid.append(i)

    if all(y) or not any(y):
        print("Not enough valid labels")
        raise Exception

    max_num_feature = max(len(f) for f in X_features)
    correct_record_idx = [i for i in range(len(X_features)) if len(X_features[i]) == max_num_feature]
    valid = [valid_idx for i, valid_idx in enumerate(valid) if len(X_features[i]) == max_num_feature]

    X_features = [f for f in X_features if len(f) == max_num_feature]
    X = np.array(X_features)
    y = np.array(y)[correct_record_idx]
    
    return X, y, valid

def stats_at_fpr(y_true, y_proba, fpr_targets, clf_step, data):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    results = {}
    batch_energy_wastage = data[~data["agent_output_is_correct"]]["total_energy"].sum()
    num_pos = y_true.sum()

    for target in fpr_targets:
        valid = np.where(fpr <= target)[0]

        if len(valid) == 0:
            results[target] = {
                "tpr": 0.0,
                "thr": None,
                "energy_wastage": batch_energy_wastage,
                "energy_wastage_reduction": 0.0,
                "avg_energy_wastage": batch_energy_wastage / num_pos,
                "avg_energy_wastage_reduction": 0.0,
                "energy_wastage_reduction_pct": 0.0,
            }
        else:
            best_idx = valid[np.argmax(tpr[valid])]
            best_thr = thr[best_idx]

            preds = (y_proba >= best_thr).astype(np.bool)
            false_neg = (y_true & (~preds)).astype(np.bool)

            energy_wastage = data[preds][f"exit_energy_step_{clf_step - 1}"].sum() + data[false_neg]["total_energy"].sum()
            energy_wastage_reduction = batch_energy_wastage - energy_wastage

            results[target] = {
                "tpr": tpr[best_idx],
                "thr": best_thr,
                "num_false_pos": np.sum(preds & ~y_true),
                "energy_wastage": energy_wastage,
                "energy_wastage_reduction": energy_wastage_reduction,
                "avg_energy_wastage": energy_wastage / num_pos,
                "avg_energy_wastage_reduction": energy_wastage_reduction / num_pos,
                "energy_wastage_reduction_pct": 100 * energy_wastage_reduction / batch_energy_wastage,
            }

    return results

def train_classifier(
    logprobs_data,
    tokens_data,
    all_data,
    labels,
    clf_step,
    n_folds=5,
    num_logprobs=10,
):
    num_total_negative = len(labels) - sum(labels)
    X, y, valid = extract_features(
        logprobs_data,
        tokens_data,
        labels,
        clf_step,
        num_logprobs=num_logprobs,
    )

    num_total = len(y)
    num_positive = y.sum()
    num_negative = num_total - num_positive

    if min(num_positive, num_negative) <= n_folds:
        return
    
    total_energy_wastage = all_data[~all_data["agent_output_is_correct"]]["total_energy"].sum()
    valid_data = all_data.iloc[valid]
    
    print(f"X dimension: {X.shape}")
    print(f"y positive: {y.mean():.3f} ({num_positive}/{num_total})")

    base_estimator = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    param_dist = {
        "max_depth": stats.randint(3, 8),
        "learning_rate": stats.loguniform(0.005, 0.2),
        "min_child_weight": stats.randint(1, 8),
        "subsample": stats.uniform(0.7, 0.3),
        "colsample_bytree": stats.uniform(0.7, 0.3),
        "gamma": stats.uniform(0, 5),

    }

    search = HalvingRandomSearchCV(
        estimator=base_estimator,
        param_distributions=param_dist,
        scoring="roc_auc",
        n_jobs=-1,
        cv=n_folds,
        factor=3,
        resource="n_estimators",
        max_resources=300,
        min_resources=10,
        random_state=42,
        verbose=0,
    )

    outer_cv = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=42,
    )

    oof_proba = np.zeros(len(y))

    fold_data = []

    print("\nRunning Nested CV...")

    for fold, (tr_idx, va_idx) in enumerate(outer_cv.split(X, y), 1):
        X_train, y_train = X[tr_idx], y[tr_idx]
        X_val, y_val = X[va_idx], y[va_idx]

        search.fit(X_train, y_train)

        best_model = search.best_estimator_

        # print("Best params:", search.best_params_)

        train_proba = best_model.predict_proba(X_train)[:, 1]
        val_proba = best_model.predict_proba(X_val)[:, 1]

        oof_proba[va_idx] = val_proba

        train_auc_score = roc_auc_score(y_train, train_proba)
        fold_auc_score = roc_auc_score(y_val, val_proba)
        
        data_val = valid_data.iloc[va_idx]
        stats_at_fpr_dict = stats_at_fpr(y_val, val_proba, fpr_targets, clf_step, data_val)

        fold_data.append({
            "fold": fold,
            "train_idx": tr_idx,
            "fold_idx": va_idx,
            "train_label": y_train,
            "fold_label": y_val,
            "train_prob": train_proba,
            "fold_prob": val_proba,
            "train_auc_score": train_auc_score,
            "fold_auc_score": fold_auc_score,
            "stats_at_fpr": stats_at_fpr_dict,
        })

    # -------------------------
    # Summary
    # -------------------------

    print("\n" + "=" * 50)
    print("FINAL NESTED CV RESULTS")
    print("=" * 50)

    train_aucs = [f["train_auc_score"] for f in fold_data]
    mean_train_auc = np.mean(train_aucs)
    std_train_auc = np.std(train_aucs)

    aucs = [f["fold_auc_score"] for f in fold_data]
    mean_fold_auc = np.mean(aucs)
    std_fold_auc = np.std(aucs)

    print(f"Train ROC-AUC: {mean_train_auc:.4f} ± {std_train_auc:.4f}")
    print(f"Test ROC-AUC: {mean_fold_auc:.4f} ± {std_fold_auc:.4f}")

    print("TPR and Energy Wastage @ FPR summary:")
    agg_fpr_stats = []
    t_crit = stats.t.ppf((1 + 0.95) / 2, df=n_folds-1)
    for fpr in fpr_targets:
        tprs = [f["stats_at_fpr"][fpr]["tpr"] for f in fold_data]
        mean_tpr = np.mean(tprs)
        std_tpr = np.std(tprs)

        num_false_pos = [f["stats_at_fpr"][fpr]["num_false_pos"] for f in fold_data]
        mean_overall_fpr = np.sum(num_false_pos) / num_total_negative
        em_overall_fpr = t_crit * np.std([n_folds * fp / num_total_negative for fp in num_false_pos]) / math.sqrt(n_folds)

        avg_energy_wastage_reduction = [f["stats_at_fpr"][fpr]["avg_energy_wastage_reduction"] for f in fold_data]
        mean_energy_wastage_reduction = np.mean(avg_energy_wastage_reduction)
        em_energy_wastage_reduction = t_crit * np.std(avg_energy_wastage_reduction) / math.sqrt(n_folds)

        avg_energy_wastage_reduction_pct = [f["stats_at_fpr"][fpr]["energy_wastage_reduction_pct"] for f in fold_data]
        mean_energy_wastage_reduction_pct = np.mean(avg_energy_wastage_reduction_pct)
        em_energy_wastage_reduction_pct = t_crit * np.std(avg_energy_wastage_reduction_pct) / math.sqrt(n_folds)

        batch_energy_wastage_reduction = [f["stats_at_fpr"][fpr]["energy_wastage_reduction"] for f in fold_data]
        mean_overall_energy_wastage_reduction_pct = 100 * np.sum(batch_energy_wastage_reduction) / total_energy_wastage
        em_overall_energy_wastage_reduction_pct = t_crit * np.std([(100 * n_folds * v / total_energy_wastage) for v in batch_energy_wastage_reduction]) / math.sqrt(n_folds)

        print(
            f"Batch FPR {fpr:.2f} ({round(fpr * num_negative)}/{num_negative}): "
            f"Overall FPR: {mean_overall_fpr:.4f} ± {em_overall_fpr:.4f}"
            f" | TPR: {mean_tpr:.4f} ± {std_tpr:.4f} ({round(mean_tpr * num_positive)}/{num_positive})"
            f" | Avg energy wastage reduction (batch) (mWh): {mean_energy_wastage_reduction:.4f} ± {em_energy_wastage_reduction:.4f}"
            f" | Reduction pct (batch): {mean_energy_wastage_reduction_pct:.4f} ± {em_energy_wastage_reduction_pct:.4f}"
            f" | Reduction pct (overall): {mean_overall_energy_wastage_reduction_pct:.4f} ± {em_overall_energy_wastage_reduction_pct:.4f}"
        )

        agg_fpr_stats.append({
            "mean_overall_fpr": mean_overall_fpr,
            "em_overall_fpr": em_overall_fpr,
            "mean_overall_energy_wastage_reduction_pct": mean_overall_energy_wastage_reduction_pct,
            "em_overall_energy_wastage_reduction_pct": em_overall_energy_wastage_reduction_pct,
        })

    return {
        "valid_idx": valid,
        "fold_data": fold_data,
        "mean_train_auc": mean_train_auc,
        "std_train_auc": std_train_auc,
        "mean_fold_auc": mean_fold_auc,
        "std_fold_auc": std_fold_auc,
        "agg_fpr_stats": agg_fpr_stats,
    }

auc_data = {}
qa_train_res = {}
for size in ["30", "1.7"]:
    for dataset in ["frames", "simpleqa"]:
        print("="*10)
        print(f"Model: Qwen3-{size}b | Dataset: {dataset}")
        all_logprobs, all_tokens, is_correct, valid_idx, all_data = qa_data[(size, dataset)]
        labels = [not c for c in is_correct]
        train_res = []
        for num_logprobs in [10]:
            for num_steps in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
                print(f"\n*** Sequence length: {num_steps} | Num min logprobs: {num_logprobs} ***")
                res = train_classifier(all_logprobs, all_tokens, all_data, labels, clf_step=num_steps, num_logprobs=num_logprobs)
                if res is not None:
                    train_res.append(res)
                else:
                    break
        auc_data[(size, dataset)] = [(res["mean_fold_auc"], res["std_fold_auc"]) for res in train_res]
        qa_train_res[(size, dataset)] = train_res

auc_data = {
    "30B-A3B, FRAMES": auc_data[("30", "frames")],
    "1.7B, FRAMES": auc_data[("1.7", "frames")],
    "30B-A3B, SimpleQA": auc_data[("30", "simpleqa")],
    "1.7B, SimpleQA": auc_data[("1.7", "simpleqa")],
}

model_colors = {
    "30B-A3B": "#56B4E9",
    "1.7B": "#CC79A7",
}

dataset_markers = {
    "FRAMES": "o",
    "SimpleQA": "s",
}

plt.figure(figsize=(6,4))
fontsize=10

offset = -0.15
offset_delta = 0.075

for label, aucs in auc_data.items():
    model, ds = label.split(", ")
    means = [e[0] for e in aucs]
    stds = [e[1] for e in aucs]

    n = 5
    se = np.array(stds) / math.sqrt(n)

    # t critical value
    t_crit = stats.t.ppf((1 + 0.95) / 2, df=n-1)

    # margin of error
    margin_error = t_crit * se

    plt.errorbar(
        np.arange(1, len(means) + 1) + offset, means, yerr=margin_error, label=label,
        ecolor="silver", color=model_colors[model], marker=dataset_markers[ds], capsize=3,
    )
    offset += offset_delta

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

plt.xticks(np.arange(1, 11))
# plt.xlabel("Step", fontsize=fontsize)

plt.yticks([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
# plt.ylabel("ROC-AUC", fontsize=fontsize)
plt.tick_params(axis="both", labelsize=fontsize)

custom_legend = []
for label, color in model_colors.items():
    custom_legend.append(Line2D([0], [0], color=color, lw=2, label=label))
for label, marker in dataset_markers.items():
    custom_legend.append(Line2D([0], [0], color="black", marker=marker, lw=2, label=label))
plt.legend(handles=custom_legend, fontsize=fontsize, frameon=False, labelspacing=0.1, borderpad=0.0, loc="lower right")

plt.show()


In [ ]:
# Figure 9: Energy wastage reduction vs Utility drop for QA

def eval_baseline(
            logprobs_data,
            all_data,
            labels,
            clf_step,
        ):
            num_total_negative = len(labels) - sum(labels)
            total_energy_wastage = all_data[~all_data["agent_output_is_correct"]]["total_energy"].sum()
            
            min_step_logprobs = []
            mean_step_logprobs = []
            y = []
            valid = []
            for i, logprobs in enumerate(logprobs_data):
                if len(logprobs) <= clf_step:
                    continue
                step_logprobs = logprobs[clf_step - 1]
                min_step_logprobs.append(min(step_logprobs))
                mean_step_logprobs.append(np.mean(step_logprobs))
                y.append(int(labels[i]))
                valid.append(i)

            y = np.array(y)
            valid_data = all_data.iloc[valid]

            stat_dicts = {
                "min_logprob": stats_at_fpr(y, min_step_logprobs, fpr_targets, clf_step, valid_data),
                "mean_logprob": stats_at_fpr(y, mean_step_logprobs, fpr_targets, clf_step, valid_data),
                "random": stats_at_fpr(y, np.random.random(len(y)), fpr_targets, clf_step, valid_data),
            }

            agg_fpr_stats = {}
            for label, stat_dict in stat_dicts.items():
                agg_fpr_stats[label] = []
                for fpr in fpr_targets:
                    fpr_stats = stat_dict[fpr]
                    num_false_pos = fpr_stats["num_false_pos"]
                    mean_overall_fpr = np.sum(num_false_pos) / num_total_negative

                    batch_energy_wastage_reduction = fpr_stats["energy_wastage_reduction"]
                    mean_overall_energy_wastage_reduction_pct = 100 * batch_energy_wastage_reduction / total_energy_wastage

                    agg_fpr_stats[label].append({
                        "mean_overall_fpr": mean_overall_fpr,
                        "mean_overall_energy_wastage_reduction_pct": mean_overall_energy_wastage_reduction_pct,
                    })

            return agg_fpr_stats

for size in ["30", "1.7"]:
    for dataset in ["frames", "simpleqa"]:
        print("="*10)
        print(f"Model: Qwen3-{size}b | Dataset: {dataset}")
        all_logprobs, all_tokens, is_correct, valid_idx, all_data = qa_data[(size, dataset)]
        labels = [not c for c in is_correct]

        # AgentStop

        plt.figure(figsize=(6, 4))

        x = np.linspace(0, 100, 100)
        plt.plot(x, x, color="silver", linestyle='--', alpha=0.5)

        for i, res in enumerate(qa_train_res[(size, dataset)]):
            fpr_stats = res["agg_fpr_stats"]
            fprs = [100 * s["mean_overall_fpr"] for s in fpr_stats]
            # fprs = [fpr for fpr in fprs if fpr <= 20]
            # fprs_em = [s["em_overall_fpr"] for s in fpr_stats]
            energy_reductions = [s["mean_overall_energy_wastage_reduction_pct"] for s in fpr_stats][:len(fprs)]
            energy_reductions_em = [s["em_overall_energy_wastage_reduction_pct"] for s in fpr_stats][:len(fprs)]

            # plt.errorbar(fprs, energy_reductions, yerr=energy_reductions_em, ecolor="silver", label=f"Step {i + 1}", marker=".", capsize=3)
            plt.plot(fprs, energy_reductions, label=f"Step {i + 1}", marker=".", linewidth=3, alpha=0.8)

        ax = plt.gca()
        ax.set_axisbelow(True)
        ax.grid(axis="both", linestyle="--", linewidth=0.8, alpha=0.6)

        # plt.xlabel("Task Performance Drop %")
        # plt.ylabel("Energy Wastage Reduction %")
        plt.xlim(-1, 26)
        plt.ylim(-1, 26)
        plt.yticks([0, 5, 10, 15, 20, 25])
        plt.xticks([0, 5, 10, 15, 20, 25])
        plt.tick_params(axis="both", labelsize=20)
        plt.show()

        # Baseline
        baseline_res = []
        for num_steps in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
            res = eval_baseline(all_logprobs, all_data, labels, clf_step=num_steps)
            baseline_res.append(res)

        plt.figure(figsize=(6, 4))

        # Default
        x = np.linspace(0, 100, 100)
        plt.plot(x, x, color="silver", linestyle='--', alpha=0.5)

        linestyle_dict = {
            "random": "--",
            "min_logprob": "dotted",
            "mean_logprob": "-.",
        }

        colors = {}

        method_n = 0
        for method, linestyle in linestyle_dict.items():
            method_n += 1
            for i, fpr_stat_methods in enumerate(baseline_res):
                fpr_stats = fpr_stat_methods[method]
                fprs = [100 * s["mean_overall_fpr"] for s in fpr_stats]
                energy_reductions = [s["mean_overall_energy_wastage_reduction_pct"] for s in fpr_stats][:len(fprs)]
                if method_n == 1:
                    line, = plt.plot(fprs, energy_reductions, linestyle=linestyle_dict[method], linewidth=3, alpha=0.7)
                    colors[i] = line.get_color()
                else:
                    plt.plot(fprs, energy_reductions, linestyle=linestyle_dict[method], linewidth=3, alpha=0.7, color=colors[i])

        ax = plt.gca()
        ax.set_axisbelow(True)
        ax.grid(axis="both", linestyle="--", linewidth=0.8, alpha=0.6)

        # plt.xlabel("Task Performance Drop %")
        # plt.ylabel("Energy Wastage Reduction %")
        plt.xlim(-1, 26)
        plt.ylim(-1, 26)
        plt.yticks([0, 5, 10, 15, 20, 25])
        plt.xticks([0, 5, 10, 15, 20, 25])
        plt.tick_params(axis="both", labelsize=20)

        # Create custom legend handles for linestyles (black color)
        # linestyle_handles = [Line2D([0], [0], color='black', lw=2, linestyle=ls) 
        #                      for ls in linestyle_dict.values()]
        # linestyle_labels = list(linestyle_dict.keys())

        # # Combine handles and labels
        # handles = []
        # labels = []

        # plt.legend(linestyle_handles, linestyle_labels, fontsize=20, loc="upper left", labelspacing=0.1, borderpad=0.0, frameon=False, columnspacing=1)
        plt.show()

In [ ]:
# Figure 10: AUC-ROC for coding

def extract_features(logprobs_data, tokens_data, labels, clf_step, num_logprobs=10):
    X_features = []
    y = []
    valid = []

    for i, (logprobs, tokens) in enumerate(zip(logprobs_data, tokens_data)):
        if len(logprobs) <= clf_step:
            continue
        feature_vector = []

        probs = [np.exp(lp) for lp in logprobs[:clf_step]]
        min_logprobs = [v for p in probs for v in np.sort(p)[:num_logprobs]]
        feature_vector.extend(min_logprobs)

        # Token features
        num_tokens = [len(lp) for lp in logprobs[:clf_step]]
        feature_vector.extend(num_tokens)

        num_thought_tokens = [(step_tokens.index("<code") if "<code" in step_tokens else len(step_tokens)) for step_tokens in tokens[:clf_step]]
        feature_vector.extend(num_thought_tokens)

        # Length of longest common substring between current step and previous step
        for j in range(clf_step - 1, clf_step):
            cur_gen = "".join(tokens[j])
            prev_gen = "".join(tokens[j - 1])
            feature_vector.append(len(lcs(cur_gen, prev_gen)) / len(cur_gen))

        X_features.append(feature_vector)
        y.append(int(labels[i]))
        valid.append(i)

    if all(y) or not any(y):
        print("Not enough valid labels")
        raise Exception

    max_num_feature = max(len(f) for f in X_features)
    correct_record_idx = [i for i in range(len(X_features)) if len(X_features[i]) == max_num_feature]
    valid = [valid_idx for i, valid_idx in enumerate(valid) if len(X_features[i]) == max_num_feature]

    X_features = [f for f in X_features if len(f) == max_num_feature]
    X = np.array(X_features)
    y = np.array(y)[correct_record_idx]
    
    return X, y, valid

labels = [not c for c in swebench_is_correct]
train_res = []
for num_logprobs in [10]:
    for num_steps in np.arange(1, 21):
        print(f"\n*** Sequence length: {num_steps} | Num min logprobs: {num_logprobs} ***")
        res = train_classifier(swebench_all_logprobs, swebench_all_tokens, swebench_df, labels, clf_step=num_steps, num_logprobs=num_logprobs)
        if res is not None:
            train_res.append(res)
        else:
            break

auc_data = [(res["mean_fold_auc"], res["std_fold_auc"]) for res in train_res]

plt.figure(figsize=(6,4))
fontsize=10

means = [e[0] for e in auc_data]
stds = [e[1] for e in auc_data]

n = 5
se = np.array(stds) / math.sqrt(n)

# t critical value
t_crit = stats.t.ppf((1 + 0.95) / 2, df=n-1)

# margin of error
margin_error = t_crit * se

# plt.errorbar(np.arange(1, len(means) + 1), means, yerr=margin_error, label=label, ecolor="silver", capsize=3, marker="o", color="#56B4E9")
x = np.arange(1, len(means) + 1)

plt.plot(x, means, label=label, marker="o", color="#56B4E9")

plt.fill_between(
    x,
    means - margin_error,
    means + margin_error,
    color="#56B4E9",
    alpha=0.2   # transparency
)

plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

plt.xticks(np.arange(1, len(means) + 1))
plt.tick_params(axis="both", labelsize=fontsize)

plt.show()


In [ ]:
# Figure 11: Energy wastage reduction vs Utility drop for coding
# Note: This is slightly different from our initial submission, but the relative behavior is still preserved

plt.figure(figsize=(6, 4))

# Default
x = np.linspace(0, 100, 100)
plt.plot(x, x, color="silver", linestyle='--', alpha=0.5)

for i, res in enumerate(train_res[:10]):
    fpr_stats = res["agg_fpr_stats"]
    fprs = [100 * s["mean_overall_fpr"] for s in fpr_stats]
    # fprs = [fpr for fpr in fprs if fpr <= 25]
    # fprs_em = [s["em_overall_fpr"] for s in fpr_stats]
    energy_reductions = [s["mean_overall_energy_wastage_reduction_pct"] for s in fpr_stats][:len(fprs)]
    energy_reductions_em = [s["em_overall_energy_wastage_reduction_pct"] for s in fpr_stats][:len(fprs)]

    # plt.errorbar(fprs, energy_reductions, yerr=energy_reductions_em, ecolor="silver", label=f"Step {i + 1}", marker=".", capsize=3)
    plt.plot(fprs, energy_reductions, label=f"Step {i + 1}", marker=".", linewidth=3, alpha=0.8)


ax = plt.gca()
ax.set_axisbelow(True)
ax.grid(axis="both", linestyle="--", linewidth=0.8, alpha=0.6)

# plt.xlabel("Task Performance Drop %")
# plt.ylabel("Energy Wastage Reduction %")
plt.xlim(-1, 26)
plt.ylim(-1, 26)
plt.yticks([0, 5, 10, 15, 20, 25])
plt.xticks([0, 5, 10, 15, 20, 25])
plt.tick_params(axis="both", labelsize=20)
# plt.legend(loc="upper center", bbox_to_anchor=(0.5, 1.15), labelspacing=0.1, borderpad=0.0, ncols=5, frameon=False, columnspacing=1)

# handles, labels = ax.get_legend_handles_labels()
# fig_leg = plt.figure(figsize=(4, 0.5))
# ax_leg = fig_leg.add_subplot(111)
# ax_leg.axis("off")  # hide axes
# legend = ax_leg.legend(handles, labels, loc="center", ncol=5, frameon=False, borderpad=0.0, columnspacing=1)
# fig_leg.savefig("../figures/energy_vs_fpr_swebench_legend.pdf", bbox_inches="tight", pad_inches=0.0)
plt.show()

plt.figure(figsize=(6, 4))

# Baseline

labels = [not c for c in swebench_is_correct]
baseline_res = []
for num_steps in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    res = eval_baseline(swebench_all_logprobs, swebench_df, labels, clf_step=num_steps)
    baseline_res.append(res)

# Default
x = np.linspace(0, 100, 100)
plt.plot(x, x, color="silver", linestyle='--', alpha=0.5)

linestyle_dict = {
    "random": "--",
    "min_logprob": "dotted",
    "mean_logprob": "-.",
}

colors = {}

method_n = 0
for method, linestyle in linestyle_dict.items():
    method_n += 1
    for i, fpr_stat_methods in enumerate(baseline_res):
        fpr_stats = fpr_stat_methods[method]
        fprs = [100 * s["mean_overall_fpr"] for s in fpr_stats]
        energy_reductions = [s["mean_overall_energy_wastage_reduction_pct"] for s in fpr_stats][:len(fprs)]
        if method_n == 1:
            line, = plt.plot(fprs, energy_reductions, linestyle=linestyle_dict[method], linewidth=3, alpha=0.7)
            colors[i] = line.get_color()
        else:
            plt.plot(fprs, energy_reductions, linestyle=linestyle_dict[method], linewidth=3, alpha=0.7, color=colors[i])

ax = plt.gca()
ax.set_axisbelow(True)
ax.grid(axis="both", linestyle="--", linewidth=0.8, alpha=0.6)

# plt.xlabel("Task Performance Drop %")
# plt.ylabel("Energy Wastage Reduction %")
plt.xlim(-1, 26)
plt.ylim(-1, 26)
plt.yticks([0, 5, 10, 15, 20, 25])
plt.xticks([0, 5, 10, 15, 20, 25])
plt.tick_params(axis="both", labelsize=20)

# color_handles = [Line2D([0], [0], color=color, lw=2) for color in colors.values()]
# color_labels = [f"Step {k + 1}" for k in colors.keys()]

# Create custom legend handles for linestyles (black color)
linestyle_handles = [Line2D([0], [0], color='black', lw=2, linestyle=ls) 
                     for ls in linestyle_dict.values()]
linestyle_labels = list(linestyle_dict.keys())

# Combine handles and labels
handles = []
labels = []

plt.legend(linestyle_handles, linestyle_labels, fontsize=20, loc="upper left", labelspacing=0.1, borderpad=0.0, frameon=False, columnspacing=1)
plt.show()